# Modeling

Training and comparing Logistic Regression, Decision Tree, and CatBoost.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import load_data
from src.models import train_logistic_regression, train_decision_tree, train_catboost
from src.evaluation import evaluate_model, cross_validate_model

In [2]:
df = load_data()

X = df.drop(columns=["y"])
y = (df["y"] == "yes").astype(int)

categorical_features = X.select_dtypes(include=["string"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

In [3]:
model_lr = train_logistic_regression(X_train, y_train, categorical_features, numerical_features)
model_dt = train_decision_tree(X_train, y_train, categorical_features, numerical_features)
model_cb = train_catboost(X_train, y_train, categorical_features)

In [4]:
results = [
  evaluate_model(model_lr, X_test, y_test, "Logistic Regression"),
  evaluate_model(model_dt, X_test, y_test, "Decision Tree"),
  evaluate_model(model_cb, X_test, y_test, "CatBoost")
]

pd.DataFrame(results)

,Model,Accuracy,F1,ROC-AUC
0,Logistic Regression,0.7548,0.3732,0.7722
1,Decision Tree,0.8232,0.4196,0.7461
2,CatBoost,0.8298,0.4541,0.7912


In [5]:
cv_results = [
  cross_validate_model(train_logistic_regression, X, y, "Logistic Regression", 
             categorical_features=categorical_features, numerical_features=numerical_features),
  cross_validate_model(train_decision_tree, X, y, "Decision Tree", 
             categorical_features=categorical_features, numerical_features=numerical_features),
  cross_validate_model(train_catboost, X, y, "CatBoost", 
             categorical_features=categorical_features)
]

pd.DataFrame(cv_results)

,Model,Mean ROC-AUC,Std
0,Logistic Regression,0.7664,0.0081
1,Decision Tree,0.7474,0.0107
2,CatBoost,0.7893,0.0079


In [6]:
feature_importance = pd.DataFrame({
  "Feature": X.columns,
  "Importance": model_cb.get_feature_importance()
}).sort_values("Importance", ascending=False)

feature_importance.head(10)

,Feature,Importance
5,balance,18.520828
9,day,13.385646
10,month,12.240896
0,age,9.772519
1,job,8.869069
11,campaign,7.733155
8,contact,6.039490
3,education,5.175496
2,marital,4.436933
14,poutcome,3.698737
